# 05 LLM-size scaling study: 0.5B vs 1.5B vs 3B speech LLM (DDP, 2x T4)

In [ ]:
# Settings: GPU T4 x2, Internet ON. Persistent outputs: results/ and runs/ in /kaggle/working.
import os, shutil, subprocess
REPO_URL = "https://github.com/ByteBoss123/speaker-aware-speech-llm.git"
WORK = "/kaggle/working/speaker-aware-speech-llm"
if not os.path.exists(WORK):
    src = "/kaggle/input/speaker-aware-speech-llm"   # alternative: upload the repo as a Kaggle dataset
    if os.path.exists(src):
        shutil.copytree(src, WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
DATA = "/tmp/data"   # large ephemeral disk; /kaggle/working is capped at ~20 GB
os.environ["DATA"] = DATA

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
!bash scripts/prepare_data.sh $DATA slurp libri_test libri_train100

Same data, same optimizer steps, same global batch (32) for every LLM size; only the LLM changes

In [ ]:
STEPS = 1500

In [ ]:
!torchrun --nproc_per_node 2 scripts/sllm_train.py --data_root $DATA --llm Qwen/Qwen2.5-0.5B-Instruct --batch 8 --grad_accum 2 --max_steps {STEPS} --out_dir runs/sllm_scale_0.5B
!python scripts/sllm_eval.py --data_root $DATA --llm Qwen/Qwen2.5-0.5B-Instruct --run_dir runs/sllm_scale_0.5B --max_per_set 2000 --out results/sllm_scale_0.5B

In [ ]:
!torchrun --nproc_per_node 2 scripts/sllm_train.py --data_root $DATA --llm Qwen/Qwen2.5-1.5B-Instruct --batch 8 --grad_accum 2 --max_steps {STEPS} --out_dir runs/sllm_scale_1.5B
!python scripts/sllm_eval.py --data_root $DATA --llm Qwen/Qwen2.5-1.5B-Instruct --run_dir runs/sllm_scale_1.5B --max_per_set 2000 --out results/sllm_scale_1.5B

In [ ]:
!torchrun --nproc_per_node 2 scripts/sllm_train.py --data_root $DATA --llm Qwen/Qwen2.5-3B-Instruct --batch 4 --grad_accum 4 --max_steps {STEPS} --out_dir runs/sllm_scale_3B
!python scripts/sllm_eval.py --data_root $DATA --llm Qwen/Qwen2.5-3B-Instruct --run_dir runs/sllm_scale_3B --max_per_set 2000 --out results/sllm_scale_3B

In [ ]:
!python scripts/summarize.py results runs
!cd /kaggle/working && zip -qr results_scale.zip speaker-aware-speech-llm/results speaker-aware-speech-llm/runs -x '*.npz'